# 1. Preparation

In [1]:
import sys
sys.path.append('../../')
from model_001 import LucaQuadruple_final_dropout, fluProfiler_Config
# sys.path.append('/data/chenyihao/LucaVirusTasks_source')
# sys.path.append('/data/chenyihao/LucaVirusTasks_source/src/lucaquadruple/models')
from tqdm import tqdm
import os
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import load_embedding, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import pickle
from utilities import print_exams

In [2]:
class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def list2df(mylist, period):
    merged_rows = []
    for i in range(0, len(mylist), period):
        merged_row = []
        for j in range(period):
            merged_row += mylist[i + j]
        merged_rows.append(merged_row)

    return pd.DataFrame(merged_rows)

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask


In [3]:
device = torch.device('cuda:1')
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
new_columns = ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d','seq_a', 'seq_b', 'seq_c', 'seq_d', 
               'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'label']
data_path = '/data/chenyihao/dataset'

Crick_all = pd.read_csv(data_path + '/all.csv')
# Crick_all = pd.read_csv(data_path + '/test/test.csv')
dataframe = Crick_all.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'seq_id_c': 'first', 'seq_id_d': 'first',
                                                  'seq_type_a': 'first', 'seq_type_b': 'first', 'seq_type_c': 'first', 'seq_type_d': 'first',
                                                  'serumName': 'first', 'virusName': 'first', 'label': 'mean'}).reset_index()
Crick_all_final = dataframe[new_columns]

Artificial_all = pd.read_csv(data_path + '/Artificial_data.csv')
dataframe = Artificial_all.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'seq_id_c': 'first', 'seq_id_d': 'first',
                                                       'label': 'mean'}).reset_index()

Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
Crick_serumType = pd.concat([Crick_H1N1,Crick_H3N2])[['virusName', 'serumType']].drop_duplicates(subset=['virusName']).reset_index(drop=True)
Crick_all_final = Crick_all_final.merge(right=Crick_serumType,how='left', left_on='virusName', right_on='virusName')

train_data, test_data = train_test_split(Crick_all_final, test_size=0.1, random_state=42)
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)
train_data = pd.concat([train_data, Artificial_all], axis=0)

train_data = train_data.sample(100)
valid_data = valid_data.sample(100)
test_data = test_data.sample(100)

train_dataset = fluProfiler_Dataset(train_data)
valid_dataset = fluProfiler_Dataset(valid_data)
test_dataset = fluProfiler_Dataset(test_data)

batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [4]:
import os
from tqdm import tqdm

ddd = pd.concat([train_data, valid_data, test_data],axis=0)
# load embedding
sequence_names = pd.concat([ddd['seq_id_a'],ddd['seq_id_b'],
                            ddd['seq_id_c'],ddd['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding", files=sequence_names)
emb_dict = dict(zip(IDs, embeddings))

Loading tensor: 100%|██████████| 646/646 [00:53<00:00, 12.06file/s]


In [12]:
dd= loaded_config.to_dict()
import json

with open("/data/chenyihao/fluProfiler_source/src/Section1_BestPractice/1.6_Pretrained_model_compare/config_dict.json", "w", encoding="utf-8") as f:
    json.dump(dd, f, indent=2, ensure_ascii=False)

In [5]:
import json
with open('./config_dict.json', 'r') as f:
    config_dict = json.load(f)
with open("./args.pkl", "rb") as file:
    fluProfiler_args = pickle.load(file)


fluProfiler_config = fluProfiler_Config.from_dict(config_dict)
model = LucaQuadruple_final_dropout(config=fluProfiler_config, args=fluProfiler_args)
model.to(device)

no_decay = ["bias", "layernorm.weight", "layer_norm.weight", "layer.norm.weight"]
optimizer_grouped_parameters = [{
            "params": [p for n, p in model.named_parameters() if not any(nd in n.lower() for nd in no_decay)],
            "weight_decay": fluProfiler_args.weight_decay
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n.lower() for nd in no_decay)],
            "weight_decay": 0.0
        }
    ]
optimizer = AdamW(optimizer_grouped_parameters,
                    lr=0.00008,
                    betas=[fluProfiler_args.beta1 if fluProfiler_args.beta1 > 0 else 0.9, fluProfiler_args.beta2 if fluProfiler_args.beta2 > 0 else 0.98],
                    eps=fluProfiler_args.adam_epsilon)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=fluProfiler_args.warmup_steps, num_training_steps=fluProfiler_args.max_steps)
epochs = 100

self.encoder_type_list: [False, True, False]
encoder_idx 1 input_size: 2560


In [11]:
# with open("./config_dict.pkl", "rb") as file:
#     config = pickle.load(file)
# with open("./args.pkl", "rb") as file:
#     fluProfiler_args = pickle.load(file)
    

# fluProfiler_config = fluProfiler_Config.from_dict(config)
# fluProfiler_config.from_dict(config)

# fluProfiler_args.matrix_encoder = False # 是否加bert
# fluProfiler_args.matrix_encoder_act = False # 是否加激活函数
# fluProfiler_Config.from_dict(ddddd)

# model = LucaQuadruple_final_dropout(config=fluProfiler_config, args=fluProfiler_args)

sys.path.append('/data/chenyihao/LucaVirusTasks_source')
sys.path.append('/data/chenyihao/LucaVirusTasks_source/src/lucaquadruple/models')
# with open("/data/chenyihao/LucaVirusTasks_source/src/lucaquadruple/models/config.pkl", "rb") as file:
#     fluProfiler_config = pickle.load(file)
with open("./args.pkl", "rb") as file:
    fluProfiler_args = pickle.load(file)

# loaded_config.classifier_size = 256 * 4
# loaded_config.embedding_input_size = 2560
# loaded_config.hidden_size = 2560 # embedding进入encoder被降维到的维度
# loaded_args.matrix_fc_size = 256*4

# fluProfiler_args.matrix_encoder = False # 是否加bert
# fluProfiler_args.matrix_encoder_act = False # 是否加激活函数

# loaded_config.num_hidden_layers = 2 # embedding降维后接入几层Bert层
model = LucaQuadruple_final_dropout(config=fluProfiler_config, args=fluProfiler_args)
model.to(device)

no_decay = ["bias", "layernorm.weight", "layer_norm.weight", "layer.norm.weight"]
optimizer_grouped_parameters = [{
            "params": [p for n, p in model.named_parameters() if not any(nd in n.lower() for nd in no_decay)],
            "weight_decay": fluProfiler_args.weight_decay
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n.lower() for nd in no_decay)],
            "weight_decay": 0.0
        }
    ]
optimizer = AdamW(optimizer_grouped_parameters,
                    lr=0.00008,
                    betas=[fluProfiler_args.beta1 if fluProfiler_args.beta1 > 0 else 0.9, fluProfiler_args.beta2 if fluProfiler_args.beta2 > 0 else 0.98],
                    eps=fluProfiler_args.adam_epsilon)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=fluProfiler_args.warmup_steps, num_training_steps=fluProfiler_args.max_steps)
epochs = 100

self.encoder_type_list: [False, True, False]
encoder_idx 1 input_size: 2560


In [ ]:
test_data_H1N1 = test_data[test_data['serumType'] == 'H1N1'].sample(100)
test_data_H3N2 = test_data[test_data['serumType'] == 'H3N2'].sample(100)
test_load = pd.concat([test_data_H1N1, test_data_H3N2], axis=0)

H1N1_dataset = LucaDataset(test_data_H1N1)
H3N2_dataset = LucaDataset(test_data_H3N2)
test_dataset = LucaDataset(test_data)

H1N1_dataloader = DataLoader(H1N1_dataset, batch_size=80, shuffle=False)
H3N2_dataloader = DataLoader(H3N2_dataset, batch_size=80, shuffle=False)
Crick_test_dataloader = DataLoader(test_dataset, batch_size=80, shuffle=False)

self.encoder_type_list: [False, True, False]
encoder_idx 1 input_size: 2560


LucaQuadruple_final_dropout(
  (matrix_dropout): Dropout(p=0.1, inplace=False)
  (matrix_pooler): GlobalMaskValueAttentionPooling1D (2560 -> 2560)
  (linear): ModuleList(
    (0): ModuleList(
      (0): Linear(in_features=2560, out_features=256, bias=True)
      (1): GELU(approximate='none')
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (hidden_layer): Linear(in_features=1280, out_features=256, bias=True)
  (hidden_act): GELU(approximate='none')
  (classifier): Linear(in_features=256, out_features=1, bias=True)
  (loss_fct): MaskedMSELoss(
    (criterion): MSELoss()
  )
  (Passage_encoder): Sequential(
    (0): Embedding(5, 256)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
)

# 2.

In [6]:
num_training_steps = len(train_dataloader) * epochs
progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=4, delta=0.005, save_dir='./fuck/')

for epoch in range(epochs):
    model.train()
    loss_ls = []
    for batch in train_dataloader:
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])
        
        matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)

        strainPassCats = strainPassCats.to(device)

        labels = labels.to(device)
        
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, matrices_d=matrixs_d, 
                                     matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                     matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, 
                                     strainPassCats=strainPassCats, labels=labels)
    
        loss.backward()
        loss_ls.append(loss.item())
        optimizer.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    print(f"Train Loss: {np.mean(loss_ls)}")

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    model.eval()
    for batch in valid_dataloader:
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
        
        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

        matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)

        strainPassCats = strainPassCats.to(device)

        labels = labels.to(device)
        with torch.no_grad():
            loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                            matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                            matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                            labels=labels)
    
        loss_ls_valid.append(loss.item())
        logits_ls.append(logits.tolist())
        prediction_ls = prediction_ls + output.view(-1).tolist()
        reference_ls = reference_ls + labels.tolist()

    print_exams(reference_ls, prediction_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls)
    valid_mse = mean_squared_error(reference_ls, prediction_ls)
    valid_pearson = pearsonr(reference_ls, prediction_ls).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls).statistic

    early_stopping(valid_mse, model)


  0%|          | 6/1300 [00:03<10:35,  2.04it/s]

KeyboardInterrupt: 

In [12]:
from utilities import print_exams

test_prediction_ls = []
test_reference_ls = []
test_logits_ls = []
model.eval()
for batch in H3N2_dataloader:
    with torch.no_grad():
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
        
        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])
        
        matrixs_a = matrixs_a.to(device)
        matrixs_b = matrixs_b.to(device)
        matrixs_c = matrixs_c.to(device)
        matrixs_d = matrixs_d.to(device)

        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)
        
        strainPassCats = strainPassCats.to(device)
        
        labels = labels.to(device)
        
        with torch.no_grad():
            loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, matrices_d=matrixs_d, 
                                         matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, 
                                        strainPassCats=strainPassCats, labels=labels)
        
        test_prediction_ls = test_prediction_ls + output.view(-1).tolist()
        test_reference_ls = test_reference_ls + labels.tolist()
        test_logits_ls.append(logits.tolist())

print_exams(test_reference_ls, test_prediction_ls)

MAE:  2.454480969812721
MSE:  8.815562911367529
pearson correlation:  PearsonRResult(statistic=-0.2076921198116704, pvalue=0.038127950355366785)
spearman correlation:  SignificanceResult(statistic=-0.2228044935986929, pvalue=0.025872522984558107)
R2_score:  -1258079.0794056766


In [ ]:
test_prediction_ls = []
test_reference_ls = []
test_logits_ls = []
model.eval()
for batch in H3N2_dataloader:
    with torch.no_grad():
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
        
        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])
        
        matrixs_a = matrixs_a.to(device)
        matrixs_b = matrixs_b.to(device)
        matrixs_c = matrixs_c.to(device)
        matrixs_d = matrixs_d.to(device)

        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)
        
        strainPassCats = strainPassCats.to(device)
        
        labels = labels.to(device)
        
        with torch.no_grad():
            loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, matrices_d=matrixs_d, 
                                         matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, 
                                        strainPassCats=strainPassCats, labels=labels)
        
        test_prediction_ls = test_prediction_ls + output.view(-1).tolist()
        test_reference_ls = test_reference_ls + labels.tolist()
        test_logits_ls.append(logits.tolist())

print_exams(test_reference_ls, test_prediction_ls)

In [ ]:
from transformers.configuration_utils import PretrainedConfig

class LucaConfig(PretrainedConfig):
    def __init__(self,
                 num_labels: int = 2,
                 vocab_size: int = 39,
                 pad_token_id: int = 0,
                 seq_fc_size: int = 1024,
                 vector_fc_size: int = 1024,
                 matrix_fc_size: int = 1024,
                 loss_reduction="mean",
                 max_position_embeddings: int = 2048,
                 type_vocab_size: int = 2,
                 num_hidden_layers: int = 12,
                 directionality="bidi",
                 initializer_range=0.02,
                 intermediate_size=4096,
                 hidden_act="gelu",
                 hidden_size: int = 1024,
                 num_attention_heads: int = 16,
                 no_token_embeddings: bool = False,
                 no_position_embeddings: bool = False,
                 no_token_type_embeddings: bool = False,
                 fc_activate_func="tanh",
                 classifier_activate_func="tanh",
                 classifier_size=1024,
                 alphabet: str = "gene_prot",
                 token_dropout: bool = True,
                 attention_probs_dropout_prob=0.1,
                 hidden_dropout_prob=0.1,
                 classifier_dropout_prob=0.1,
                 ignore_index=-100,
                 pos_weight=1.0,
                 layer_norm_eps=1e-12,
                 position_embedding_type="absolute",
                 self_atten=True,
                 cross_atten=True,
                 use_luca_layer_norm_v2=True,
                 kernel_size=7,
                 **kwargs):
        super().__init__(pad_token_id=pad_token_id, **kwargs)
        self.num_labels = num_labels
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.seq_fc_size = seq_fc_size
        self.vector_fc_size = vector_fc_size
        self.matrix_fc_size = matrix_fc_size
        self.loss_reduction = loss_reduction
        self.max_position_embeddings = max_position_embeddings
        self.type_vocab_size = type_vocab_size
        self.num_hidden_layers = num_hidden_layers
        self.directionality = directionality
        self.initializer_range = initializer_range
        self.intermediate_size = intermediate_size
        self.hidden_act = hidden_act
        self.hidden_size = hidden_size
        self.num_attention_heads = num_attention_heads
        self.no_token_embeddings = no_token_embeddings
        self.no_position_embeddings = no_position_embeddings
        self.no_token_type_embeddings = no_token_type_embeddings
        self.fc_activate_func = fc_activate_func
        self.classifier_size = classifier_size
        self.alphabet = alphabet
        self.token_dropout = token_dropout
        self.attention_probs_dropout_prob = attention_probs_dropout_prob
        self.hidden_dropout_prob = hidden_dropout_prob
        self.classifier_dropout_prob = classifier_dropout_prob
        self.ignore_index = ignore_index
        self.pos_weight = pos_weight
        self.layer_norm_eps = layer_norm_eps
        self.position_embedding_type = position_embedding_type
        self.classifier_activate_func = classifier_activate_func
        self.self_atten = self_atten
        self.cross_atten = cross_atten
        self.use_luca_layer_norm_v2 = use_luca_layer_norm_v2
        self.kernel_size = kernel_size

In [ ]:
model = torch.load('/data/chenyihao/fluProfiler/model/001.pth', weights_only=False)
torch.save(model.state_dict(), '/data/chenyihao/mytest.pth')

In [ ]:
model = LucaQuadruple_final_dropout(config=loaded_config, args=loaded_args)
model.to(device)

# model2.load_state_dict(model.state_dict())
no_decay = ["bias", "layernorm.weight", "layer_norm.weight", "layer.norm.weight"]
optimizer_grouped_parameters = [{
            "params": [p for n, p in model.named_parameters() if not any(nd in n.lower() for nd in no_decay)],
            "weight_decay": loaded_args.weight_decay
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n.lower() for nd in no_decay)],
            "weight_decay": 0.0
        }
    ]
optimizer = AdamW(optimizer_grouped_parameters,
                    lr=0.00008,
                    betas=[loaded_args.beta1 if loaded_args.beta1 > 0 else 0.9, loaded_args.beta2 if loaded_args.beta2 > 0 else 0.98],
                    eps=loaded_args.adam_epsilon)
# scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=loaded_args.warmup_steps, num_training_steps=loaded_args.max_steps)
epochs = 100